# 02 IOU Results

This notebook displays quantitative IOU, Precision, Recall, F1, and segment-count results for the proposed method against manual labels. It does not run the reconstruction algorithm.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

IOU_DIR = REPO_ROOT / "data_iou"
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

print("REPO_ROOT:", REPO_ROOT)
print("IOU_DIR:", IOU_DIR)


## Proposed Method vs Manual Labels

`manual*.csv` are manual labels. `auto2019.csv`, `auto2020.csv`, etc. are the proposed method results.


In [ ]:
from ce4_lpr.metrics import evaluate_years, phase_summary

results = evaluate_years(IOU_DIR, YEARS, pred_prefix="auto", ref_prefix="manual")
display(results)

summary = phase_summary(results)
display(summary)


## Compact Paper-Style Table

This table keeps the key columns used for reporting the proposed method performance.


In [ ]:
compact = results[[
    "Year", "Precision", "Recall", "IOU", "F1_Score",
    "Manual_Count", "Pred_Count", "Count_Diff"
]].copy()
display(compact)

all_avg_f1 = summary.loc[summary["Phase"] == "All Chang'e-4 (2019-2024)", "Avg_F1"].iloc[0]
all_avg_iou = summary.loc[summary["Phase"] == "All Chang'e-4 (2019-2024)", "Avg_IOU"].iloc[0]
print("All-year average F1:", all_avg_f1)
print("All-year average IOU:", all_avg_iou)


## Plot Annual F1 and Count Difference


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ce4_lpr.plotting import apply_paper_style

apply_paper_style()
FIG_DIR = REPO_ROOT / "outputs" / "iou_results"
FIG_DIR.mkdir(parents=True, exist_ok=True)

x = np.arange(len(YEARS))
fig, ax1 = plt.subplots(figsize=(9, 5), dpi=300)
ax1.plot(x, results["F1_Score"], color="#C0392B", marker="o", linewidth=1.8, markersize=5, label="F1 Score")
ax1.set_xticks(x)
ax1.set_xticklabels(YEARS)
ax1.set_ylim(0.90, 1.00)
ax1.set_xlabel("Year")
ax1.set_ylabel("F1 Score")
ax1.grid(True, linestyle="--", linewidth=0.4)
ax1.legend(loc="lower left", frameon=False)

ax2 = ax1.twinx()
ax2.bar(x, results["Count_Diff"], width=0.35, color="#BDC3C7", alpha=0.75, edgecolor="black", linewidth=0.5, label="Count Difference")
ax2.set_ylabel("Count Difference")
ax2.legend(loc="upper right", frameon=False)

plt.title("Annual Evaluation of Valid-Segment Extraction", fontweight="bold")
plt.tight_layout()
fig_path = FIG_DIR / "annual_f1_count_difference.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
print("Saved figure:", fig_path)
plt.show()


## CE-3 Row Note

The paper also reports a CE-3 test row. The current open-source data folder contains only CE-4 yearly CSV files, so this notebook recomputes the CE-4 rows and CE-4 phase summaries. Add CE-3 manual and prediction CSV files before recomputing the CE-3 row here.
